# R2lab monolithic OCUDU gNB with a Benetel RU (O-RAN 7.2) and interference

Single OCUDU gNB on **sopnode-f3** driving the Benetel RAN550 **benetel1** over the O-RAN 7.2 fronthaul (Open Fronthaul library of OCUDU, DPDK on an SR-IOV VF, PTP time synchronization), with `qhat01`, `qhat02`, `qhat03` as physical UEs in R2lab. Only the gNB runs on a server (`sopnode-f3`, POS); the Kubernetes control plane and the iperf server, the core and its database, and the monitoring with the live CSI view run on the VMs `node1`, `node10` and `node11`.

Same node, RU and carrier as `oai-benetel`; the OCUDU radio and fronthaul parameters are those of the 5g_ansible OCUDU Benetel setup (https://github.com/turletti/srsran-helm, branch `ocudu`, `charts/srsran-gnb/values-benetel1.yaml`), with the 2.5 ms TDD pattern of the OAI notebook (the IFRAN Benetel configuration supports periods of 5 slots at most).

## The interference scenario

This notebook deploys the same network as `ocudu-benetel`, then runs an **interference scenario**, as the 5g_ansible
interference scenario does: while one UE sends or receives an iperf transfer, a USRP of R2Lab emits noise in the band
of the cell for a while, then stops. The run records three sections for each direction:

| Section | What happens |
|---|---|
| `<direction>_baseline` | the transfer runs alone: the reference |
| `<direction>_interference` | the USRP emits: start and end are known to the second |
| `<direction>_recovery` | the emission has stopped: does the link come back? |

**Why the link degrades.** The gNB (OCUDU, the Benetel RAN550 `benetel1` (sopnode-f3)) adapts the link to the signal-to-interference-plus-noise
ratio (SINR). The noise adds to the interference level `I0` measured on each resource block (RB):

- **Uplink**: the RU receives the noise with the signal of the UE. The uplink SNR measured by the gNB drops, the
  scheduler lowers the uplink MCS, the uplink BLER and the HARQ retransmissions rise, the throughput falls; the UE
  power control asks for more power (the power headroom drops).
- **Downlink**: the UE receives the noise with the signal of the cell. Its CQI reports drop, so the downlink MCS
  drops too.
- In **TDD** the uplink and the downlink use the same frequencies one after the other, and the noise is continuous:
  it hits both directions. Its effect depends on the distances between the USRP, the RU and the UE, on the gains,
  and on the width of the noise: a few MHz of noise in a 100 MHz carrier only hit a few RBs, which the scheduler
  can avoid or hardly notices.

**What measures it.**

- **Grafana** (monitoring on `node11`): the interference windows are added as annotations (shaded regions) on every
  dashboard, so the panels can be read before, during and after.
- **Prometheus export** of the run: the mean of each RAN metric per section.
- **CSI**: the channel `H` that the gNB estimates on the SRS of the UE is an uplink measurement. The noise adds to the
  estimate on the RBs it covers: their power rises in the `interference` sections only.
- **Spectrum** (optional): a fit node shows the band, as snapshots (power spectral density before and during the
  emission) or live (`uhd_fft` in a VNC session, as the 5g_ansible `watch` role does).

**The pieces.**

| Piece | Where | How |
|---|---|---|
| Noise generator, wide (default) | USRP N320 or N300 of R2Lab, driven by the server `sopnode-f1` (POS) | `rhubarbe pdu on`, R2Lab netplan and UHD 4.7 + GNU Radio built on the server (`usrp_noise` playbook), then `uhd_siggen` |
| Noise generator, narrow | USRP of fit02 (B210) or fit08 (B205mini) | `rhubarbe load -i gnuradio`, `rhubarbe usrpon`, then `uhd_siggen` |
| Spectrum view (optional) | a fit node not used for the noise | `uhd_rx_cfile` + numpy (snapshots) or `uhd_fft` in VNC (live) |
| Control | this notebook, over SSH (R2Lab gateway `faraday.inria.fr` with your R2Lab slice, POS bastion for the server) | `r2lab_experiments.interference` |

**Which generator.** The noise of a fit node USRP stays narrow (15 MHz in 5g_ansible): in a 100 MHz carrier, hardly
anything shows. The N320 and the N300 emit tens of MHz, up to the whole carrier. They are on the R2Lab USRP network
(VLAN `r2lab_usrp`; N300 at 192.168.235.103, N320 at 192.168.235.105), which a SophiaNode server reaches with the
R2Lab netplan, and they need UHD 4.7: the server `sopnode-f1` is provisioned for that, joins the cluster like the other nodes but runs no
5G component, and builds UHD 4.7 and GNU Radio (as the 5g_ansible `uhd-gnuradio` role does) in the background once
Kubernetes is deployed: count tens of minutes. The sample rate sets the width of the noise, and the USRP can only
use its master clock divided by an integer: 245.76 MHz on the N320 (61.44, 49.152, 40.96 or 30.72 MHz), the default
master clock on the N300; UHD takes the nearest rate, and `uhd_siggen -v` prints it.

IFRAN does not drive these USRPs: its R2Lab provisioner only powers the RUs and the UEs. They are driven over SSH
with the commands of 5g_ansible, with an active R2Lab lease for your slice, as for the UEs.

**The frequency plan** of the cell: n78, 100 MHz: 273 RB of 360 kHz from point A (ARFCN 626772, 3401.58 MHz) to 3499.86 MHz, SSB (ARFCN 630048) at the carrier center, 3450.72 MHz. The presets of the configuration cell:

| Preset | Center, width (Hz) | Where |
|---|---|---|
| `data_wide` | 3477.50e6, 40.96e6 | upper part of the carrier (RB 154 to 267), away from the SSB: N300/N320 |
| `full` | 3450.72e6, 122.88e6 | the whole carrier and the SSB: N320 (the UE drops) |
| `data` | 3475.00e6, 15e6 | upper half of the carrier (RB 183 to 224), away from the SSB: fit node USRP |
| `ssb` | 3450.72e6, 15e6 | on the SSB and the carrier center: synchronization and broadcast hit, the UE may drop |
| `outside` | 3530.00e6, 15e6 | above the carrier: control run, no effect expected |

**The waveforms** of `uhd_siggen`: `gaussian` (white Gaussian noise over the sample rate, the 5g_ansible choice),
`uniform` (uniform noise), `sine` (one tone at `freq + waveform_freq`), `const` (a pure carrier at `freq`), `2tone`
(two tones) and `sweep` (a tone sweeping the band). Noise hits every RB it covers; a tone hits one RB.

First, we install the required dependencies:

## Install Dependencies:

Install the SLICES CLI used for the BI to create VMs, and for POS to provision baremetal servers:

In [ ]:
%pip install slices-cli --extra-index-url=https://doc.slices-ri.eu/pypi/

Install the SLICES POS CLI:

In [ ]:
%pip install slices-cli-pos --index-url https://gitlab.inria.fr/api/v4/projects/68732/packages/pypi/simple --extra-index-url https://doc.slices-ri.eu/pypi/

Authenticate with your SLICES account:

In [ ]:
#! slices auth login

Install IFRAN:

In [ ]:
from pathlib import Path
import os
import sys

REPOSITORY = Path("../..").resolve()
# This repository (r2lab_experiments: O-RAN 7.2 charts and templates, handover trigger) and NADOR as published
# (nador==1.1.1, from PyPI): install it in editable mode.
%pip install -e "{REPOSITORY}" --extra-index-url=https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple
# IFRAN develop (UE configure, RU SSH access, kubelet parameters, miscellaneous data in the rendered files), not yet in
# an IFRAN release: the O-RAN 7.2 node setup and the Benetel RU configuration are playbooks of this repository.
# Install it after this repository, whose NADOR requirement (ifran==0.0.6) would otherwise replace it.
# To test unpushed IFRAN changes, set IFRAN to the path of a local clone. Do not install it in editable
# mode (-e): the IFRAN Ansible roles are only found in an installed package.
IFRAN = "ifran @ git+https://github.com/ifran-project/ifran.git@develop"
# IFRAN = "/Users/<user>/git/post-5g/ifran"
# Refresh upstream even when its package version number has not changed.
%pip install --upgrade --force-reinstall --no-deps "{IFRAN}"
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]


Install the Ansible dependencies:

In [ ]:
!{sys.executable} -m ansible.cli.galaxy install -r ../../ansible/requirements.yaml

Make sure you have an SSH key that you add to SLICES using the command:

```bash
slices pubkey register /path/to/ssh/key
```

which will be use to grant SSH access to the provisioned SLICES machines.

## Input 5G Network Description:

IFRAN keeps the logical network description separate from the deployment-specific choices and evolving deployment state, so the same experiment can be adapted to different implementations or testbeds.

To achieve that, it uses two separate models:
- The Post-5G metamodel describing the entire network and its relationships using in pure 3GPP-standard way
- The deployment model describing implementation choices, target nodes, provider settings, and operational state

To deploy, you invoke lifecycle managers for the work you need. For example, the `ProvisioningManagement` class reserves or creates resources, `KubernetesManagement` prepares nodes and forms a cluster, `CoreManagement`, `RANManagement`, `RICManagement`, and `UEManagement` prepare and install workloads on their respective 5G components.

These stages are composable management operations, not necessarily one automatic "deploy everything" command.

Each stage exposes its particular set of actions that run the corresponding Ansible roles and Helm actions, often concurrently where possible.

Edit the SLICES project, the VM infrastructure, flavor and lifetime here. Keep the SLICES CLI authenticated in this environment.

In [ ]:
from datetime import UTC, datetime

# VMs of the experiment (BI provisioner); the experiment name includes a timestamp, so that a run never reuses
# (and deletes) the VMs of another one
PROJECT = "post5g-beta"
EXPERIMENT = "ocudu-benetel-interference-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
INFRA = "fr-sophia2-bi-vm1"
VM_FLAVOR = "large"
VM_DURATION = "8h"  # VM lifetime
VM_NODES = ["node1", "node10", "node11"]
print(f"Experiment: {EXPERIMENT}")


In [ ]:
from metamodel_3gpp.models.metamodel import MetaModel

# Radio parameters of the 5g_ansible / oai5g-rru Benetel gNB (rru/gnb-config-benetel.yaml):
# n78, 100 MHz (273 PRB), SSB 630048, pointA 626772, TDD 2.5 ms (3 DL slots + 6 DL symbols, 1 UL slot + 4 UL symbols).
# OCUDU: 4T2R (4 DL antennas, 2 UL antennas, Benetel mimo_mode 1_2_3_4_4x2) and PRACH configuration 159, as in
# the OCUDU RAN550 profile; same carriers and 2.5 ms TDD pattern as the OAI notebook.
metamodel = MetaModel.from_yaml(
"""
rus:
  benetel1:
    type: real
    timeSource: external
    clockSource: external
    numberOfTxPorts: 4
    numberOfRxPorts: 2
    txAttenuationDB: 0
    rxAttenuationDB: 0
    rxGainDB: 75
ran:
  components:
    gnb:
      type: gnb
      config:
        radioConfiguration:
          nrBand: 78
          channelBandwidthMhz: 100
          downlinkArfcn: 630048  # carrier centre, used by OCUDU (dl_arfcn)
          pointAFrequencyArfcn: 626772
          ssbArfcn: 630048
          ssbPbchBlockPowerDbm: -15
          prachConfigurationIndex: 159
          msg1FrequencyStartPrb: 0
          tddConfiguration:
            periodicityMilliseconds: 2.5
            nrofDownlinkSlots: 3
            nrofDownlinkSymbols: 6
            nrofUplinkSlots: 1
            nrofUplinkSymbols: 4
        nrCellId: 1
        physicalCellId: 1
        ru: benetel1
        gnbId: 1
ric:
  components: {}
ues:
  qhat01:
    type: cots
    config:
      supi: '001010000000006'
      authenticationData:
        k: fec86ba6eb707ed08905757b1bb44b8f
        opc: C42449363BBAD02B66D16BC975D77CC1
      gnbs:
      - gnb
  qhat02:
    type: cots
    config:
      supi: '001010000000007'
      authenticationData:
        k: fec86ba6eb707ed08905757b1bb44b8f
        opc: C42449363BBAD02B66D16BC975D77CC1
      gnbs:
      - gnb
  qhat03:
    type: cots
    config:
      supi: '001010000000008'
      authenticationData:
        k: fec86ba6eb707ed08905757b1bb44b8f
        opc: C42449363BBAD02B66D16BC975D77CC1
      gnbs:
      - gnb
"""
)

To see the full content of this Meta Model:

In [ ]:
print(metamodel.to_yaml(exclude_defaults=False))

Next, to get the deployment model, we need initiate a `DeploymentModel` object.

The easiest way to do so is by providing a YAML input that describes our deployment.

Three VMs (BI provisioner, `INFRA` above): `node1` runs the Kubernetes control plane and the iperf server, `node10` the core and its database, `node11` the monitoring and the live CSI view. `sopnode-f3`, provisioned by POS, runs the gNB (the `legacy_pos` provisioner remains declared but no node uses it). `FHI72_NODE_PARAMS` (after the deployment model) describes the O-RAN 7.2 setup of `sopnode-f3`: fronthaul PF and its VFs (one per C/U plane, with the DU MAC and the RU VLAN), housekeeping CPUs, NIC IRQ pinning and PTP.

The deployment model needs your [R2Lab](https://r2lab.inria.fr/tuto-010-registration.md) credentials (your R2Lab slice name, email, and password).

A good practice to prevent private information leak (e.g., password) is to store the sensitive information in a special location. In this, we keep it as environment variables that we store in a `.env` [file](https://pypi.org/project/python-dotenv/), next to this notebook or in a parent directory (e.g. the root of this repository, shared by all the notebooks; `load_dotenv()` takes the first one found going up from the notebook directory):

```
R2LAB_SLICE=<YOUR_R2LAB_SLICE>
R2LAB_EMAIL=<YOUR_R2LAB_EMAIL>
R2LAB_PASSWORD=<YOUR_R2LAB_PASSWORD>
```

The following lines load variables from this `.env` file. Make sure to never commit your secrets in public repositories.

In [ ]:
%pip install python-dotenv

In [ ]:
import json
import os
from dotenv import load_dotenv

load_dotenv()
R2LAB_SLICE = os.getenv("R2LAB_SLICE")
R2LAB_EMAIL = os.getenv("R2LAB_EMAIL")
R2LAB_PASSWORD = os.getenv("R2LAB_PASSWORD")
missing = [name for name in ("R2LAB_SLICE", "R2LAB_EMAIL", "R2LAB_PASSWORD") if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Set {missing} in a .env file next to this notebook or in a parent directory")

### Interference configuration

Edit the scenario here, before the deployment model: an N300/N320 generator adds its server to the model.
`INTERFERENCE = False` runs the same sections without emission: a reference run, to tell the effect of the noise
from the variations of the radio channel. The gain is the TX gain of the USRP: UHD keeps it within the range of the
USRP (5g_ansible gives 110 dB, the maximum is lower) and `uhd_siggen -v` prints the value set. Lower it, or the
amplitude, for a partial degradation.

In [ ]:
from r2lab_experiments.interference import FIT_USRPS, N3XX_USRPS, FrequencyPlan

# Carrier of the cell (metamodel above): point A, number of RBs, subcarrier spacing, SSB
PLAN = FrequencyPlan(point_a_arfcn=626772, n_rb=273, scs_khz=30, ssb_arfcn=630048)
# Center frequency and width of the interference (Hz)
PRESETS = {
    "data_wide": (3477.50e6, 40.96e6),  # upper part of the carrier (RB 154 to 267), away from the SSB: N300/N320
    "full": (3450.72e6, 122.88e6),  # the whole carrier and the SSB: N320 (the UE drops)
    "data": (3475.00e6, 15e6),  # upper half of the carrier (RB 183 to 224), away from the SSB: fit node USRP
    "ssb": (3450.72e6, 15e6),  # on the SSB and the carrier center: synchronization and broadcast hit, the UE may drop
    "outside": (3530.00e6, 15e6),  # above the carrier: control run, no effect expected
}
PRESET = "data_wide"

INTERFERENCE = True  # False: the same sections without emission (reference run)
NOISE_USRP = "n320"  # "n320", "n300" (driven by NOISE_SERVER), "b210" (fit02) or "b205mini" (fit08)
NOISE_SERVER = "sopnode-f1"  # POS server driving the N320/N300, not used by the 5G network
NOISE_FREQ, NOISE_RATE = PRESETS[PRESET]  # center and width (sample rate of uhd_siggen)
NOISE_GAIN = 60 if NOISE_USRP in N3XX_USRPS else 89  # TX gain (dB)
NOISE_WAVEFORM = "gaussian"  # gaussian, uniform, sine, const, 2tone, sweep
NOISE_AMPLITUDE = 0.15  # baseband amplitude, 0 to 1
NOISE_WAVEFORM_FREQ = 0.0  # tone offset from NOISE_FREQ (sine, 2tone, sweep), Hz

SPECTRUM = "snapshots"  # "off", "snapshots" (spectrum before and during the emission) or "live" (VNC uhd_fft)
SPECTRUM_USRP = "b210" if NOISE_USRP != "b210" else "b205mini"  # fit node USRP of the spectrum view
SPECTRUM_RATE, SPECTRUM_GAIN = 40e6, 30  # band shown around NOISE_FREQ (Hz) and RX gain (dB)

N3XX = NOISE_USRP in N3XX_USRPS
NOISE_FIT = FIT_USRPS.get(NOISE_USRP)
SPECTRUM_FIT = FIT_USRPS[SPECTRUM_USRP]
if SPECTRUM_FIT == NOISE_FIT:
    raise ValueError("the spectrum view needs a fit node other than the noise generator")
# The deployment model below includes the noise server when an N320/N300 emits
NOISE_SERVER_NODE = f"  {NOISE_SERVER}:\n    provisioner: pos\n    roles: []\n" if N3XX else ""
print(PLAN.describe(NOISE_FREQ, NOISE_RATE))

We can now create the deployment model.

In [ ]:
from pathlib import Path

from ifran.models.deployment import DeploymentModel

# OCUDU images: IFRAN defaults for a Benetel RU (registry.gitlab.com/ocudu/ocudu/images: du-dpdk or gnb-dpdk, cu-cp,
# cu-up); the csi metric of monitoring.yaml replaces the gNB image by the CSI logger build
# (r2labuser/ocudu-gnb-csi:release_26_10) for the radios that log.

# VMs (BI provisioner): Kubernetes control plane and iperf server (node1), core and database (node10), monitoring
# and live CSI view (node11). sopnode-f3 (POS) runs the gNB.
deployment_model = DeploymentModel.from_yaml(
f"""
params:
  provisioning:
    extra_path: ''
  k8s:
    master_node: node1
    local_kubeconfig_path: {Path(f".k8s-{EXPERIMENT}.conf").resolve()}
provisioners:
  bi:
    type: bi
    params:
      project: {PROJECT}
      experiment: {EXPERIMENT}
      infra: {INFRA}
      flavor: {VM_FLAVOR}
      create_experiment: true
      node_duration: {VM_DURATION}
  pos:
    type: pos
    params:
      attempt_reservation: false
  legacy_pos:
    type: legacy_pos
    params:
      attempt_reservation: false
      central_pos_ssh_info:
        hostname: duckburg.net.cit.tum.de
        port: 10022
        user: <YOUR_SLICES_USERNAME>
  r2lab:
    type: r2lab
    params:
      r2lab_slice: {json.dumps(R2LAB_SLICE)}
      reservation_params:
        email: {json.dumps(R2LAB_EMAIL)}
        password: {json.dumps(R2LAB_PASSWORD)}
nodes:
{NOISE_SERVER_NODE}  node1:
    provisioner: bi
    roles: []
  node10:
    provisioner: bi
    roles:
    - core
    - database
  node11:
    provisioner: bi
    roles: []
  sopnode-f3:
    provisioner: pos
    # du: the fhi72 and benetel playbooks run on the du_nodes group (radio node of the gNB)
    roles:
    - ran
    - du
    provisioning_params:
      # PREEMPT_RT kernel image, as with 5g_ansible (pos_rt_image)
      image: ubuntu-questing-rt@2025-10-29T19:34:30+00:00
      is_staged_image: true
      # 5g_ansible boot_parameters_rt_f3: housekeeping CPUs 0-5 (= fhi72 reserved_cpus), 40 x 1G hugepages
      boot_params: net.ifnames=1 biosdevname=0 isolcpus=managed_irq,6-63 nohz_full=6-63
        nohz=on rcu_nocbs=6-63 kthread_cpus=0-5 irqaffinity=0-5 rcu_nocb_poll
        selinux=0 enforcing=0 intel_pstate=disable nosoftlockup
        hugepagesz=1G hugepages=40 hugepagesz=2M hugepages=0 default_hugepagesz=1G
        mitigations=off intel_iommu=on processor.max_cstate=1 idle=poll
        intel_idle.max_cstate=0 iommu=pt skew_tick=1 tsc=nowatchdog
        nmi_watchdog=0 softlockup_panic=0 audit=0 mce=off nosmt
    # static CPU manager for the Guaranteed gNB pod, system and Kubernetes processes on the housekeeping CPUs
    k8s_params:
      kubelet_params:
        cpu_manager_policy: static
        cpu_manager_policy_options:
          full-pcpus-only: 'true'
        reserved_system_cpus: 0-5
        topology_manager_policy: best-effort
ran:
  implementation: ocudu
  components:
    gnb:
      type: gnb
      node_name: sopnode-f3
      advanced_config:
        # Open Fronthaul (O-RAN 7.2) to the Benetel RAN550, as in the 5g_ansible OCUDU Benetel setup (srsran-helm
        # charts/srsran-gnb/values-benetel1.yaml): one VF (U-plane VF of the node, MAC 00:11:22:33:44:66) for both planes.
        # @FHI72_...@ placeholders are replaced by the OCUDU chart of this repository when the pod starts (VF PCI address, pinned CPUs).
        ru_ofh:
          t1a_max_cp_dl: 470
          t1a_min_cp_dl: 419
          t1a_max_cp_ul: 336
          t1a_min_cp_ul: 285
          t1a_max_up: 345
          t1a_min_up: 294
          ta4_max: 200
          ta4_min: 0
          is_prach_cp_enabled: true
          compr_method_ul: bfp
          compr_bitwidth_ul: 9
          compr_method_dl: bfp
          compr_bitwidth_dl: 9
          compr_method_prach: bfp
          compr_bitwidth_prach: 9
          enable_ul_static_compr_hdr: false
          enable_dl_static_compr_hdr: false
          iq_scaling: 5.5
          cells:
          - network_interface: "@FHI72_PCI@"
            mtu: 9216
            ru_mac_addr: 8c:1f:64:d1:12:8c
            du_mac_addr: 00:11:22:33:44:66
            vlan_tag_cp: 801
            vlan_tag_up: 801
            prach_port_id: [4, 5]
            dl_port_id: [0, 1, 2, 3]
            ul_port_id: [0, 1]
        prach:
          prach_config_index: 159
          prach_root_sequence_index: 1
          zero_correlation_zone: 0
          # contention-free random access (handover): preambles 60-63 are kept out of contention-based access
          cfra_enabled: true
          total_nof_ra_preambles: 64
          nof_cb_preambles_per_ssb: 60
        # uplink power of the 5g_ansible setup: closed loop, so that the UEs near the RU do not saturate its input
        pucch:
          p0_nominal: -90
          enable_cl_loop_pw_control: true
        pusch:
          p0_nominal_with_grant: -76
          enable_cl_loop_pw_control: true
          target_sinr: 15
          max_rank: 1
        # PHY at info writes a line per PUCCH/PDSCH/PUSCH (99% of the log): kubelet then rotates the 10 MB log files
        # within seconds; RLC, PDCP and GTP-U at info write lines for every packet (MAC, scheduler and the other layers
        # stay at info)
        log:
          filename: stdout
          all_level: info
          phy_level: warning
          rlc_level: warning
          pdcp_level: warning
          gtpu_level: warning
          config_level: info
        # remote control server and JSON metrics: set by ran_kpis of monitoring.yaml (configure_ran_monitoring)
        # threads on the CPUs pinned by the static CPU manager: CPU 0 for DPDK (hal eal_args, set by nador), 1 for the
        # OFH timing, 2 for the OFH Tx/Rx, 3 to 5 for the main pool
        expert_execution:
          affinities:
            main_pool_cpus: "@FHI72_CPU_3@,@FHI72_CPU_4@,@FHI72_CPU_5@"
            main_pool_pinning: mask
            ofh:
              timing_cpu: "@FHI72_CPU_1@"
              txrx_cpus: "@FHI72_CPU_2@"
          threads:
            main_pool:
              nof_threads: 3
              task_queue_size: 4096
              backoff_period: 50
            upper_phy:
              pdsch_processor_type: auto
              pdsch_cb_batch_length: 4
              max_pusch_and_srs_concurrency: 2
              max_pucch_concurrency: 2
              max_pdsch_concurrency: 2
            lower_phy:
              execution_profile: triple
      # DPDK: privileged container (VF bound to vfio-pci, hugepages)
      security_context:
        privileged: true
        capabilities:
          add: [IPC_LOCK, SYS_NICE, NET_ADMIN]
      # Guaranteed pod (requests = limits, integer cpu): the static CPU manager pins its CPUs
      resources:
        define: true
        requests:
          nf:
            cpu: 8
            memory: 8Gi
        limits:
          nf:
            cpu: 8
            memory: 8Gi
rus:
  benetel1:
    vendor: benetel
    model: ran550
    provisioner: r2lab
    # no Python on the RU: the benetel playbook only runs raw commands there
    ssh_info:
      extra_ansible_vars:
        ansible_python_interpreter: ''
    ips:
    - 192.168.233.101
ues:
  qhat01:
    type: cots
    provisioner: r2lab
  qhat02:
    type: cots
    provisioner: r2lab
  qhat03:
    type: cots
    provisioner: r2lab
"""
)

# O-RAN 7.2 node and Benetel RU, outside IFRAN: miscellaneous field of the deployment model, read by the fhi72 and
# benetel playbooks of this repository (ansible/) and by its OCUDU charts (SR-IOV resources, 1G hugepages)
from r2lab_experiments.fhi72 import BenetelRUParams, Fhi72NodeParams, fhi72_miscellaneous

# DU MAC addresses and VLAN of the VFs, also loaded in the RU (/etc/ru_config.cfg)
FHI72_NODE_PARAMS = {
    "sopnode-f3": Fhi72NodeParams.model_validate({
        "fronthaul_interface": "ens15f1np1",
        "vfs": [
            {"resource_name": "ruvfiou", "plane": "U", "mac": "00:11:22:33:44:66", "vlan": 801},
            {"resource_name": "ruvfioc", "plane": "C", "mac": "00:11:22:33:44:67", "vlan": 801},
        ],
        "reserved_cpus": "0-5",
        "irq_pattern": "ice-|ens15f[0-3]np[0-3]",
        "ptp": {"interface": "ens15f2np2"},
    }),
}
# ru_config.cfg of the 5g_ansible OCUDU profile (ru_profiles/benetel/config/benetel1-srsran-...cfg): dynamic
# compression, no long-form PRACH compression, and the C-plane sent to the single DU VF
BENETEL = BenetelRUParams(
    tx_power_dbm=24, downlink_scaling=0, compression="dynamic_compressed",
    extra={"c_plane_du_mac": "00:11:22:33:44:66", "lf_prach_compression_enable": "false"},
)
# RU -> RAN node of its gNB (which reaches its management IP), gNB, parameters
BENETEL_RUS = {"benetel1": ("sopnode-f3", "gnb", BENETEL)}
deployment_model.miscellaneous = fhi72_miscellaneous(metamodel, FHI72_NODE_PARAMS, BENETEL_RUS, hugepages_1gi="2Gi")

> *IMPORTANT NOTE*: Make sure your R2Lab credentials are not shared publicly!

By default, the metamodel and deployment model will use Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR
and one 5G slice (`internet`) with one DNN (`internet`).

## **1.**   Provisioning nodes, RUs and UEs:

In [ ]:
from ifran.deployment import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

With the nodes, IFRAN powers on the UEs and the RU (`benetel1`) through the radio API, in parallel, then waits until the API reports the RU ready. The API checks that an RU answers ping on its management network (192.168.233.0/24 for the Benetel RUs): the radio initialization of the RU is checked later by the `benetel` playbook.

In [ ]:
# Nodes (POS servers, VMs), RUs and UEs: provisioned in parallel
await prov.provision()

### Fit nodes

The fit nodes (spectrum view, and the noise generator with `NOISE_USRP = "b210"` or `"b205mini"`) load the GNU Radio
image now, in the background, while the deployment goes on: `rhubarbe load` of this image takes several minutes.
Their USRP is then powered on. The scenario waits for them before it starts.

In [ ]:
import asyncio

from r2lab_experiments.interference import prepare_fit_node

# Background task: the next cells run while the fit nodes load their image
FIT_NODES = ([NOISE_FIT] if INTERFERENCE and not N3XX else []) + ([SPECTRUM_FIT] if SPECTRUM != "off" else [])
fit_preparation = asyncio.ensure_future(asyncio.gather(*(prepare_fit_node(R2LAB_SLICE, fit) for fit in FIT_NODES)))
print("Preparing fit nodes:", [f"fit{fit:02d}" for fit in FIT_NODES])

### R2Lab netplan of sopnode-f3

As with 5g_ansible: DHCP on the PTP interface (`ptp`, which also carries the RU management network 192.168.233.0/24) gives `sopnode-f3` access to the RU (`ssh benetel1`), and the USRP and AW2S VLANs on the fronthaul PF (`rrus`). Only the netplan role of the r2lab example tuning is applied: the fhi72 setup below replaces its NIC and CPU tuning. It is applied right after provisioning, before Kubernetes and the fhi72 setup: `netplan apply` resets the interfaces, and the RU management network is needed by the `benetel` playbook.

In [ ]:
from pathlib import Path

from ifran.deployment import MiscellaneousManagement

misc_net = MiscellaneousManagement(prov)
await misc_net.run_playbook(
    playbook=str(Path("netplan.yaml").resolve()),
    roles_paths=[str(Path("../oai-aw2s/tuning/ansible/roles").resolve())],
)

## **2.**   Kubernetes deployment, RU and UE configuration:

The RU and the UEs are configured while Kubernetes is deployed: all three are independent and take minutes.

Meanwhile, the `benetel` playbook of this repository loads the OCUDU configuration of `benetel1` (`/etc/ru_config.cfg`, the `ru_config` of `miscellaneous`) over SSH at its management IP, through `sopnode-f3` (raw commands: the RU has no Python): DU MAC addresses and VLAN of the VFs, centre frequency, bandwidth and TDD pattern of the gNB radio configuration, MIMO mode of the RU ports, and the `BENETEL` parameters. The RU is then rebooted (a few minutes), and the playbook waits for its radio initialization.

In [ ]:
import asyncio

from ifran.deployment import K8SManagement, MiscellaneousManagement, UEManagement

k8s = K8SManagement(prov)

# Benetel RU: the benetel playbook reaches it over SSH at its management IP (provisioned address), through the RAN
# node of its gNB (jumphost)
misc_ru = MiscellaneousManagement(preceding=prov)
for ru_name, (node_name, _, _) in BENETEL_RUS.items():
    ru = misc_ru.deployment_model.rus[ru_name]
    ru.ssh_info.hostname = str(ru.ips[0])
    ru.ssh_info.jumphost = misc_ru.deployment_model.nodes[node_name].ssh_info

# The UEs are configured (provisioner); the core and RAN models start from the Kubernetes one, so the configured
# state is copied there
ue_setup = UEManagement(preceding=prov)
await asyncio.gather(
    k8s.deploy(),
    misc_ru.run_playbook(
        playbook=str(REPOSITORY / "ansible" / "playbooks" / "benetel.yaml"),
        roles_paths=[str(REPOSITORY / "ansible" / "roles")],
        retries=1,  # an RU reboot takes minutes: no retry
    ),
    ue_setup.configure(),
)
for name, ue_prov in ue_setup.deployment_model.ues.items():
    k8s.deployment_model.ues[name].connection_status = ue_prov.connection_status

### Noise server

With an N320/N300 generator, now that Kubernetes is deployed (no more package installation on the nodes), the
`usrp_noise` playbook of this repository prepares `NOISE_SERVER` in the background: R2Lab netplan (the `r2lab_usrp`
VLAN reaches the USRPs), then UHD 4.7 and GNU Radio built from source (tens of minutes). Meanwhile, the R2Lab gateway
powers the USRP on (`rhubarbe pdu on`). Once the build is done, `uhd_find_devices` waits for the USRP.

In [ ]:
import asyncio

from ifran.deployment import MiscellaneousManagement
from r2lab_experiments.interference import node_ssh_command, power_n3xx, wait_usrp


async def prepare_noise_server():
    misc_noise = MiscellaneousManagement(k8s)
    await asyncio.gather(
        power_n3xx(R2LAB_SLICE, NOISE_USRP, on=True),
        misc_noise.run_playbook(
            playbook=str(REPOSITORY / "ansible" / "playbooks" / "usrp_noise.yaml"),
            roles_paths=[str(REPOSITORY / "ansible" / "roles"), str(Path("../oai-aw2s/tuning/ansible/roles").resolve())],
            inputs={"noise_node": NOISE_SERVER},
            retries=1,  # a build failure must show at once
        ),
    )
    return await wait_usrp(node_ssh_command(k8s, NOISE_SERVER), N3XX_USRPS[NOISE_USRP])


noise_server_setup = asyncio.ensure_future(prepare_noise_server()) if N3XX else None
print(f"Preparing {NOISE_SERVER} for the {NOISE_USRP}" if N3XX else "No noise server: fit node generator")

## **3.**   O-RAN 7.2 fronthaul setup and 5G core deployment:

Both are independent and run in parallel: the fronthaul setup on the RAN node, the core on the VMs.

Once `sopnode-f3` is in the cluster: VFs created on `ens15f1np1` and bound to `vfio-pci`, NIC IRQs pinned on the housekeeping CPUs, CPU governor, RCU threads, sysctl, SR-IOV device plugin (resources `openshift.io/ruvfioc` and `openshift.io/ruvfiou`) and PTP (`ptp4l` slave on `ens15f2np2`, domain 24, and `phc2sys`), by the `fhi72` playbook of this repository (`ansible/`), with the `fhi72_params` of `miscellaneous`. The kubelet static CPU manager (`full-pcpus-only`) is set when the node joins the cluster (`k8s_params` of the node).

In [ ]:
from ifran.deployment import CoreManagement

misc = MiscellaneousManagement(k8s)
core = CoreManagement(k8s)


async def deploy_core() -> None:
    await core.run_database_action(action="prepare")
    await core.prepare()
    # Preserve the database and subscriber steps performed by upstream core.deploy().
    await core.run_database_action(action="install")
    await core.install()
    await core.register_ues()


# O-RAN 7.2 setup of the du_nodes (fhi72_params of the miscellaneous field; it needs the kubelet) and core on the VMs
await asyncio.gather(
    misc.run_playbook(
        playbook=str(REPOSITORY / "ansible" / "playbooks" / "fhi72.yaml"),
        roles_paths=[str(REPOSITORY / "ansible" / "roles")],
        # no retry: a failure (e.g. an unreachable RAN node) shows up at once instead of after 5 full runs
        retries=1,
    ),
    deploy_core(),
)

Check the node: 40 free 1G hugepages, both VFs with their MAC and VLAN 801 bound to `vfio-pci`, `ptp4l` and `phc2sys` active, and a PTP grandmaster present with a small `master_offset` (ns).

In [ ]:
# Fronthaul node state: 1G hugepages, VFs (MAC/VLAN) bound to vfio-pci, PTP (ptp4l slave, phc2sys)
check = await prov.run_node_command(
    command=["bash", "-c", " ; ".join([
        "grep -E 'HugePages_(Total|Free)|Hugepagesize' /proc/meminfo",
        "ip -d link show ens15f1np1 | grep -E '^ +vf '",
        "for vf in /sys/class/net/ens15f1np1/device/virtfn*; do "
        "echo $(basename $(readlink $vf)) $(basename $(readlink $vf/driver)); done",
        "systemctl is-active ptp4l phc2sys",
        "/usr/local/sbin/pmc -u -b 0 -d 24 'GET TIME_STATUS_NP' | grep -E 'master_offset|gmPresent|gmIdentity'",
        "true",  # print every check, even when one fails
    ])],
    nodes=["sopnode-f3"],
    become=True,
    quiet=True,
)
print(check)


In [ ]:
# The SR-IOV device plugin must advertise one VF per resource, and the kubelet the 1G hugepages
allocatable = await prov.run_node_command(
    command=["kubectl", "--kubeconfig", "/etc/kubernetes/admin.conf", "get", "node", "sopnode-f3",
             "-o", "jsonpath={.status.allocatable}"],
    nodes=["node1"],
    become=True,
    quiet=True,
)
print(allocatable)


## **4.**   RAN deployment:

### Deploy the monolithic gNB on sopnode-f3

The gNB pod requests the two VFs (it uses the U-plane one) and the 1G hugepages of `sopnode-f3`, and pins its threads on the CPUs allocated by the static CPU manager (`expert_execution` and DPDK EAL placeholders, replaced at start by the chart of this repository).

In [ ]:
from ifran.deployment import RANManagement

# RU and UEs provisioned in section 1
prov_radio = ProvisioningManagement(core)
ran = RANManagement(prov_radio)

In [ ]:
# OCUDU charts of this repository: O-RAN 7.2 fronthaul of the gNB/DU (SR-IOV VF, 1G hugepages, VF and CPU
# placeholders replaced at start). Their templates are set after configure_ran_monitoring (use_ocudu_fhi72_files):
# NADOR 1.1.1 refuses templates set before on a component that logs CSI.
from r2lab_experiments import OCUDU_CHARTS

ocudu_charts = OCUDU_CHARTS
for name, component in ran.deployment_model.ran.components.items():
    if component.implementation == "ocudu" and component.type in ("gnb", "du"):
        component.custom_chart_path = str(ocudu_charts / f"ocudu-{component.type}")

### Monitoring profile and CSI logging

`monitoring.yaml` sets what the monitoring collects. Its `ran_kpis` and `csi` metrics change the RAN, so the profile is read now and `configure_ran_monitoring` applies them before the RAN is deployed:

- `ran_kpis: {ocudu_port: 8001}`: each OCUDU component streams its JSON metrics on its remote control server (port 8001), which the `ocudu-metrics` chart of the monitoring reads.
- `csi`: The OCUDU gNB (benetel1, `sopnode-f3`) can log SRS channel estimates per resource block. The `csi` metric of `monitoring.yaml` (`components: [gnb]`) turns the logger on: `gnb` gets the OCUDU image with the CSI logger (`r2labuser/ocudu-gnb-csi`, built from `ocudu-csi-logging`), periodic SRS in its configuration (`srs_period_ms`), and the logger output in `/mnt/csi` on the node (the only hostPath of the OCUDU charts): each logger writes `/mnt/csi/csi_per_rb.csv`. The logger options (`granularity`, `subcarrier_sampling`, `antennas`, `ports`, `srs_period_ms`) are those of the `csi` metric: see the [CSI guide](../../docs/guide/csi.md).

With a real radio the file grows by several MB/s: experiments copy only the per-section files (the `CsiCollector` options below), still reading the whole file on the radio node at the lowest priority, on its housekeeping CPUs. Run these cells before preparing the radios.

#### What the CSI is

The CSI logged here is the raw uplink channel `H` that the gNB estimates on the sounding reference signal (SRS) of each UE: one complex value (amplitude and phase) per resource block, receive antenna and SRS port, at every SRS occasion. A CSI logger patched into the PHY of the gNB copies each estimate into memory, and a separate thread writes it every 5 seconds to a CSV file on the radio node. NADOR shows this file live and collects it per experiment section.

The [CSI guide](../../docs/guide/csi.md) explains why the SRS is used, how the OAI and OCUDU gNBs extract it, what NADOR deploys, shows and collects, the tools that read the files, and the limits of the approach.

In [ ]:
from nador import MonitoringDeployment, configure_ran_monitoring

# The same profile is used below to deploy the monitoring
config = MonitoringDeployment.from_yaml_file("monitoring.yaml")
configure_ran_monitoring(ran, config)
# NADOR 1.1.1 renders the OCUDU radios that log CSI with its own templates: use this repository's, which apply the
# same CSI logger SRS and the O-RAN 7.2 edits
from r2lab_experiments import use_ocudu_fhi72_files

print("O-RAN 7.2 templates:", use_ocudu_fhi72_files(ran))

# RAN components that log CSI (the csi metric of monitoring.yaml)
CSI_COMPONENTS = [
    name for name, component in ran.deployment_model.ran.components.items()
    if str((component.extra_env_vars or {}).get("CSI_LOGGING_ENABLED")) == "1"
]
for name in CSI_COMPONENTS:
    component = ran.deployment_model.ran.components[name]
    print(f"{name} on {component.node_name}: {component.image.repository}:{component.image.image_tag}",
          component.extra_env_vars)

The R2Lab nodes boot a live image: their root filesystem is in RAM, where the CSI file would fill the memory
(5g_ansible saw the gNB pod killed after about 2 minutes). `prepare_csi_storage` bind-mounts the `csi-data`
directory of the node's data disk onto `/mnt/csi`, as 5g_ansible does; the partitions are those of the 5g_ansible
inventory (`storage=`). The disk is never formatted, and the mount does not survive a reboot: rerun this cell after
reprovisioning. It stops with an explicit error if a node's directory is in RAM and no partition is given.

In [ ]:
from nador.csi import prepare_csi_storage

# Data disk partition (under /dev) of each radio node whose root is in RAM; nodes on a disk need no entry
CSI_STORAGE = {'sopnode-f3': 'sdb2'}  # 5g_ansible deploy.sh get_storage
for directory, report in (await prepare_csi_storage(ran, CSI_STORAGE)).items():
    print(f"{directory}: {report['before']}  ->  {report['after']}")

In [ ]:
await ran.prepare()

In [ ]:
await ran.install()

Check the gNB logs:

In [ ]:
print(await ran.get_logs(quiet=True))

The gNB is deployed and ready for UE connections.

## **5.**   Connecting UEs:

## Monitoring before UE attachment

Deploy monitoring on the VM **node11** before connecting the physical UEs: the CPUs of the RAN node are isolated for the gNB.

In [ ]:
from pathlib import Path
import asyncio
import json

from nador import MonitoringManagement

QUERIES_FILE = Path("queries.json")  # Your Prometheus export query profile.

# config: the monitoring profile read before the RAN deployment (monitoring.yaml)
monitoring = MonitoringManagement(preceding=ran, config=config)


In [ ]:
try:
    await monitoring.deploy()
finally:
    monitoring.output(
        metamodel_file="monitoring_metamodel.yaml",
        deployment_model_file="monitoring_deployment_model.yaml",
    )

### Open Grafana

The SSH tunnel uses the current model address and jump host.
Open the link after deployment; close the tunnel with `grafana_tunnel.terminate()`.

In [ ]:
import atexit
import shlex
import subprocess
from IPython.display import Markdown, display
from ifran.models.provisioning import SSHInfo
from ifran.utils.utils import deep_merge

# Close the previous tunnel before opening one with the current host address.
if "grafana_tunnel" in globals() and grafana_tunnel.poll() is None:
    grafana_tunnel.terminate()
    await asyncio.to_thread(grafana_tunnel.wait, 5)

# get SSH info and fallback
cfg = monitoring.deployment_model.monitoring
control = monitoring.deployment_model.params.k8s.master_node
# global SSH defaults merged with the SSH info of the node, as for Ansible
ssh = SSHInfo.model_validate(deep_merge(
    monitoring.deployment_model.params.ssh.model_dump(mode="json", exclude_unset=True),
    monitoring.deployment_model.nodes[control].ssh_info.model_dump(mode="json", exclude_unset=True),
))

command = ["ssh", "-N", "-T", "-S", "none", "-o", "ControlMaster=no", "-o", "ControlPersist=no",
           "-o", "BatchMode=yes", "-o", "ExitOnForwardFailure=yes", "-o", "ConnectTimeout=10"]

command += shlex.split(ssh.get_common_args() or "")

if ssh.private_key_file:
    command += ["-i", str(Path(ssh.private_key_file).expanduser())]
command += ["-p", str(ssh.port), "-L", f"127.0.0.1:30096:127.0.0.1:{cfg.grafana_node_port}",
            f"{ssh.user}@{ssh.hostname or control}"]

grafana_tunnel = subprocess.Popen(command, stdout=subprocess.DEVNULL)

# Close the tunnel when the kernel exits normally.
atexit.register(grafana_tunnel.terminate)

GRAFANA_URL = "http://127.0.0.1:30096"
display(Markdown(f"[Open Grafana]({GRAFANA_URL}) — username: **admin**, password: **admin**"))

### Live CSI view

One Streamlit view per CSI radio runs on `node11`, never on a radio node: it follows the radio node's CSI
file over SSH, where only `sshd` and `tail -F` run, at the lowest priority on the housekeeping CPUs `0-4`. The view
shows a sliding window (120 s by default) in local time. Start it once the radio runs (it reads the file header);
the tunnel is opened on this machine. Stop it with the cell before the UE disconnection.

In [ ]:
from nador.csi import CsiLiveView
from IPython.display import Markdown, display

# Live view on node11, never on a radio node; housekeeping CPUs for "tail -F" on the radio node
CSI_LIVE = {"window_s": 120, "refresh_s": 10, "max_mb": 1024, "cpu_list": "0-4"}
CSI_LIVE_PORTS = {'gnb': 8501}
lives = globals().get("lives", {})
for name in CSI_COMPONENTS:
    if name in lives:
        await lives[name].close_tunnel()  # re-run: free the local port of the previous tunnel
    lives[name] = CsiLiveView(ran, live_node='node11', component=name, port=CSI_LIVE_PORTS[name], **CSI_LIVE)
    url = await lives[name].start()
    display(Markdown(f"[CSI {name}]({url})"))

In [ ]:
from ifran.deployment import UEManagement

ue = UEManagement(preceding=ran)

In [ ]:
await ue.connect()

Check the gNB logs after connecting the UEs:

In [ ]:
print(await ran.get_logs(quiet=True))

Check that each CSI logger runs and that its file grows on disk: the logs show the logger start, and each
`csi_per_rb.csv` must grow between the two listings (the logger flushes every 5 seconds); `df` shows the space left.

In [ ]:
from ifran.deployment import ProvisioningManagement

for name in CSI_COMPONENTS:
    logs = await ran.get_logs(components=[name], quiet=True)
    print(f"--- {name}")
    print("\n".join(line for line in logs.splitlines() if "CSI" in line))

csi_nodes = {name: ran.deployment_model.ran.components[name].node_name for name in CSI_COMPONENTS}
nodes = ProvisioningManagement(preceding=ran)
for _ in range(2):  # sizes must grow between the two listings
    listing = await nodes.run_node_command(
        command=["sh", "-c", "ls -l /mnt/csi/csi_per_rb.csv; df -h /mnt/csi | tail -1"],
        nodes=sorted(set(csi_nodes.values())), become=True, quiet=True, flatten=False,
    )
    for name, node in csi_nodes.items():
        print(f"{name} ({node}):\n{listing[node].strip()}")
    await asyncio.sleep(10)

The 3 UEs are connected to the 5G network.

Let's verify their connectivity by pinging the UPF:

In [ ]:
ping = await prov_radio.run_ue_command(
    command=["ping", "-c", "6", "11.1.1.1", "-I", "wwan0"], quiet=True, flatten=False
)


In [ ]:
print(ping["qhat01"])

In [ ]:
print(ping["qhat02"])

In [ ]:
print(ping["qhat03"])

## Interference generator and spectrum

Wait for the fit nodes and the noise server: each prints the USRP that `uhd_find_devices` finds.

In [ ]:
for fit, found in zip(FIT_NODES, await fit_preparation):
    print(f"--- fit{fit:02d}\n{found}")
if noise_server_setup is not None:
    print(f"--- {NOISE_SERVER}: {NOISE_USRP}\n{await noise_server_setup}")

The generator runs `uhd_siggen` on the noise server (N320/N300) or on the fit node, through SSH. `start()` returns once the USRP emits (`uhd_siggen`
prints its prompt once its flow graph runs), and `stop()` answers the prompt: the emission window is known to the
second. A `timeout` on the fit node stops the emission anyway after `safety_s` seconds, should this notebook stop
first. The command run on the fit node:

In [ ]:
from IPython.display import Markdown, display

from r2lab_experiments.interference import UHD_ENVIRONMENT, LiveSpectrum, NoiseGenerator, spectrum_snapshot

# N320/N300: uhd_siggen on the noise server, with the UHD 4.7 it built; fit node USRP: on the fit node
server = {}
if N3XX:
    server = {"ssh": node_ssh_command(k8s, NOISE_SERVER), "usrp_args": N3XX_USRPS[NOISE_USRP], "environment": UHD_ENVIRONMENT}
noise = NoiseGenerator(
    R2LAB_SLICE, NOISE_FIT or 0, freq=NOISE_FREQ, rate=NOISE_RATE, gain=NOISE_GAIN, waveform=NOISE_WAVEFORM,
    amplitude=NOISE_AMPLITUDE, waveform_freq=NOISE_WAVEFORM_FREQ, **server,
)
print(noise.command())

live_spectrum = None
if SPECTRUM == "live":
    # apt installs VNC and XFCE on the fit node (minutes), then the tunnel opens on this machine
    live_spectrum = LiveSpectrum(R2LAB_SLICE, SPECTRUM_FIT, freq=NOISE_FREQ, rate=SPECTRUM_RATE, gain=SPECTRUM_GAIN)
    url = await live_spectrum.start()
    display(Markdown(f"Live spectrum: open `{url}` in a VNC client (macOS: Finder > Go > Connect to Server), "
                     f"password `{live_spectrum.password}`"))

### Check the emission

A short emission alone, before the scenario: with `SPECTRUM = "snapshots"`, the other fit node takes a spectrum
before and during it. The noise must stand out of the floor over `NOISE_RATE` around `NOISE_FREQ`; the cell signal
shows too when it falls in the band shown. The values are relative (dB): the USRPs are not calibrated. With
`SPECTRUM = "live"`, watch the VNC window instead.

In [ ]:
%pip install matplotlib numpy

In [ ]:
import matplotlib.pyplot as plt


async def snapshot():
    return await spectrum_snapshot(R2LAB_SLICE, SPECTRUM_FIT, freq=NOISE_FREQ, rate=SPECTRUM_RATE, gain=SPECTRUM_GAIN)


def plot_spectra(spectra: dict, title: str):
    fig, ax = plt.subplots(figsize=(10, 4))
    for label, s in spectra.items():
        ax.plot([f / 1e6 for f in s["freq_hz"]], s["psd_db"], label=label, linewidth=0.8)
    ax.axvspan((NOISE_FREQ - NOISE_RATE / 2) / 1e6, (NOISE_FREQ + NOISE_RATE / 2) / 1e6, alpha=0.1, color="red",
               label="interference band")
    for edge in (PLAN.low_hz, PLAN.high_hz):
        ax.axvline(edge / 1e6, color="gray", linestyle="--", linewidth=0.8)
    ax.set(xlabel="MHz", ylabel="relative dB", title=title)
    ax.legend()
    plt.show()


if SPECTRUM == "snapshots":
    check = {"before": await snapshot()}
    if INTERFERENCE:
        await noise.start()
        try:
            check["during"] = await snapshot()
        finally:
            await noise.stop()
    plot_spectra(check, f"fit{SPECTRUM_FIT:02d} around {NOISE_FREQ / 1e6:.2f} MHz")
elif INTERFERENCE:
    await noise.start()
    await asyncio.sleep(10)
    await noise.stop()
    print("\n".join(noise.output[-10:]))

## Interference scenario with experiment artifacts

One UE (`CLIENT`), one direction after the other. For each direction, one iperf transfer covers the three sections:

1. the transfer starts, and gets a few seconds to settle (outside the sections);
2. `<direction>_baseline`: `BASELINE_S` seconds without emission;
3. the USRP starts (a few seconds, outside the sections), then `<direction>_interference`: `INTERFERENCE_S` seconds
   of emission; the emission stops at the end of the section, and the window becomes a Grafana annotation;
4. `<direction>_recovery`: `RECOVERY_S` seconds without emission.

The sections last 40 seconds: the CSI logger writes its file every 5 seconds with 1-second time markers, and the
batches at the edge of a section belong to both sections. With `SPECTRUM = "snapshots"`, a spectrum is taken in each
section. The collector then exports Prometheus and cuts the CSI file per section, as in the other notebooks.

In [ ]:
from datetime import UTC, datetime
from ifran.deployment import ProvisioningManagement
from nador.artifacts import ExperimentRun
from nador.csi.collection import CsiCollector
from nador.traffic import Iperf

SERVER_NODE = monitoring.deployment_model.params.k8s.master_node
CLIENT = "qhat01"  # the UE of the scenario
DIRECTIONS = ["uplink", "downlink"]
UE_INTERFACE = "wwan0"
BASELINE_S, INTERFERENCE_S, RECOVERY_S = 40, 40, 40
SETTLE_S = 10  # transfer start before the baseline section
IPERF_S = SETTLE_S + BASELINE_S + INTERFERENCE_S + RECOVERY_S + 30  # 30 s: USRP start and margin
BASE_PORT, COLLECTION_SECONDS = 5202, 5

# COTS UE over SSH: iperf3 installed where missing, address of the UE on UE_INTERFACE
clients = ProvisioningManagement(preceding=ue)
iperf = await Iperf.setup(clients, server=SERVER_NODE, interface=UE_INTERFACE, clients=[CLIENT])
print(f"Server: {SERVER_NODE} ({iperf.server_ip}); UE client: {iperf.ue_ips}")

In [ ]:
# CSI retrieval by each ExperimentRun: the CSI file is cut per section on the radio node, at the lowest priority on
# the housekeeping CPUs, and only the cuts are fetched (the whole file grows by several MB/s)
CSI_COLLECTORS = [CsiCollector(monitoring, name, fetch_full=False, cpu_list="0-4") for name in CSI_COMPONENTS]

In [ ]:
SCENARIO = [
    {"name": f"{CLIENT}_{direction}", "client": CLIENT, "direction": direction, "duration": IPERF_S}
    for direction in DIRECTIONS
]
print(json.dumps(SCENARIO, indent=1))

In [ ]:
from nador.artifacts.loki_export import OCUDU_PHY_DETAIL

# Loki export: leave out the per-slot UL/DL scheduling decisions and the MAC PDU lines, nearly all the exported lines
LOKI_DROP = [
    OCUDU_PHY_DETAIL,
    r"Slot decisions pci=\d+ t=\d+us \([^)]*\): (UL|DL): ",
    r"\] UL rnti=0x[0-9a-f]+ ue=\d+ subPDUs: ",
    r"\] DL PDU: ue=\d+ rnti=0x[0-9a-f]+ size=\d+: ",
]
from r2lab_experiments.interference import annotate_grafana

timestamp = datetime.now(UTC).strftime("%Y-%m-%d_%H-%M-%S.%fZ")
output_directory = Path("results") / f"interference-{timestamp}"
spectra, windows = {}, []


async def section(name: str, seconds: float):
    """One section of `seconds`; with snapshots, the other fit node takes a spectrum meanwhile."""
    with experiment.section(name):
        wait = asyncio.ensure_future(asyncio.sleep(seconds))
        if SPECTRUM == "snapshots":
            try:
                spectra[name] = await snapshot()
            except RuntimeError as exc:  # the spectrum is optional: the section goes on
                print(f"spectrum {name}: {exc}")
        await wait


async with ExperimentRun(
    output_directory,
    name="interference",
    monitoring=monitoring,
    queries_file=QUERIES_FILE,
    collectors=CSI_COLLECTORS,
    loki_drop=LOKI_DROP,
) as experiment:
    experiment.save_json("scenario.json", SCENARIO)
    experiment.save_json("interference.json", {
        "enabled": INTERFERENCE, "preset": PRESET, "usrp": NOISE_USRP, "command": noise.command(),
        "freq_hz": NOISE_FREQ, "rate_hz": NOISE_RATE, "plan": PLAN.describe(NOISE_FREQ, NOISE_RATE),
    })
    for port, test in enumerate(SCENARIO, start=BASE_PORT):
        direction = test["direction"]
        print(f"Running {test['name']}: {CLIENT}, {direction}, {IPERF_S} seconds")
        await iperf.serve(port)
        transfer = asyncio.ensure_future(iperf.client(CLIENT, direction, IPERF_S, port))
        await asyncio.sleep(SETTLE_S)
        await section(f"{direction}_baseline", BASELINE_S)
        if INTERFERENCE:
            start = await noise.start()
        try:
            await section(f"{direction}_interference", INTERFERENCE_S)
        finally:
            if INTERFERENCE:
                stop = await noise.stop()
                windows.append({"direction": direction, "start": start, "end": stop})
        if INTERFERENCE and "GRAFANA_URL" in globals():
            text = f"{NOISE_WAVEFORM} {NOISE_FREQ / 1e6:.2f} MHz, {NOISE_RATE / 1e6:g} MHz wide, gain {NOISE_GAIN} dB"
            try:
                await asyncio.to_thread(annotate_grafana, GRAFANA_URL, start, stop, text, tags=["interference", direction])
            except OSError as exc:  # the annotation is optional
                print(f"Grafana annotation: {exc}")
        await section(f"{direction}_recovery", RECOVERY_S)
        report = await transfer
        # a failed test returns a report with an error key: save it and go on with the next test
        if report.get("error"):
            print(f"FAILED {test['name']}: {report['error']}")
        experiment.save_json(f"{test['name']}.json", report)
    experiment.save_json("interference_windows.json", windows)
    experiment.save_json("spectra.json", spectra)

    with experiment.section("collection"):
        await asyncio.sleep(COLLECTION_SECONDS)

print(f"Artifacts saved to: {output_directory.resolve()}")

In [ ]:
from nador.artifacts import iperf_summary

print(iperf_summary(output_directory))

Check the CSI collected by the run: per radio, its node clock (the offset to this machine is only applied above
0.5 s), the rows and 5-second batches read in the whole file, and the per-section files under
`csi/<component>/by_window/section/`.

In [ ]:
meta = json.loads((output_directory / "experiment_metadata.json").read_text())
print(meta["status"], meta["collection_errors"])
for name, m in meta.get("csi", {}).items():  # empty without CSI
    print(f"--- {name} on {m['node']}")
    print("  clock:", {k: m.get(k) for k in ("offset_applied_s", "clock_drift_s", "warnings")})
    print("  start:", {k: round(m["clock_start"][k], 3) for k in ("offset_s", "rtt_s", "uncertainty_s")})
    print("  data: ", {k: m.get("summary", {}).get(k) for k in ("rows", "markers", "dropped_rows", "windows_written", "windows_empty")})
print(sorted(str(p.relative_to(output_directory)) for p in (output_directory / "csi").rglob("*.gz")))

## Analysis

### Throughput

The throughput of each iperf transfer, second by second, with the sections. The iperf intervals are on the clock of
the UE and the sections on the clock of this machine: both are NTP synchronized, but check the offset when a change
seems early or late by a few seconds.

In [ ]:
from r2lab_experiments.interference import iperf_intervals, section_windows

sections = section_windows(output_directory)
colors = {"baseline": "green", "interference": "red", "recovery": "blue"}
fig, axes = plt.subplots(len(SCENARIO), 1, figsize=(10, 3 * len(SCENARIO)), squeeze=False)
for ax, test in zip(axes[:, 0], SCENARIO):
    report = json.loads((output_directory / f"{test['name']}.json").read_text())
    times, mbps = iperf_intervals(report)
    if not times:
        ax.set_title(f"{test['name']}: no iperf interval ({report.get('error', 'no report')})")
        continue
    t0 = times[0]
    ax.plot([t - t0 for t in times], mbps, color="black", linewidth=1)
    for name, (start, end) in sections.items():
        kind = name.rsplit("_", 1)[-1]
        if name.startswith(test["direction"]) and kind in colors:
            ax.axvspan(start - t0, end - t0, alpha=0.12, color=colors[kind], label=kind)
    ax.set(title=f"{test['name']} ({test['direction']})", xlabel="s", ylabel="Mbit/s")
    ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

### Grafana

The interference windows are annotations (shaded regions) on every dashboard. On the OCUDU dashboard, read the
**PUSCH SNR** and **PUCCH SNR** (the uplink SNR measured by the gNB: it drops on the noise), the uplink and downlink
**BLER** and **MCS**, the **CQI** reported by the UE, and the bitrates.

The links open the dashboards on the time of the run.

In [ ]:
from r2lab_experiments.interference import grafana_dashboards

campaign = json.loads((output_directory / "timeline_summary.json").read_text())["windows"][0]
time_range = f"from={int(campaign['start_epoch'] * 1000)}&to={int(campaign['end_epoch'] * 1000)}"
if "GRAFANA_URL" in globals():
    links = [f"- [{title}]({GRAFANA_URL}/d/{uid}?{time_range})" for title, uid in grafana_dashboards(GRAFANA_URL).items()]
    display(Markdown("\n".join(links)))
print(json.dumps(windows, indent=1))

### RAN metrics per section

The mean of each RAN metric exported from Prometheus, per section (`METRIC_FILTER`: words of the series to keep). A
metric that moves in the `interference` sections and comes back in the `recovery` sections shows the effect of the
noise; compare with a reference run (`INTERFERENCE = False`) to tell it from the variations of the channel.

In [ ]:
from r2lab_experiments.interference import prometheus_section_means

METRIC_FILTER = "snr,bler,nok,mcs,cqi,brate"
means = prometheus_section_means(output_directory, metric_filter=METRIC_FILTER)
order = [name for name in sections if name.rsplit("_", 1)[-1] in colors]
print(f"{'series':70s}" + "".join(f"{name:>22s}" for name in order))
for series, per_section in means.items():
    print(f"{series[:70]:70s}" + "".join(f"{per_section.get(name, float('nan')):22.2f}" for name in order))

### CSI per resource block

The mean power of the channel estimated on the SRS, per RB and per section (all SRS occasions, antennas and ports of
the section). The SRS is sent by the UE: this is an uplink measurement. The noise adds to the estimate, so the RBs it
covers (shaded) rise in the `interference` sections, and come back in the `recovery` sections. Outside the noise band,
the curves follow the channel and the UE power control. The values are relative (see the CSI guide).

In [ ]:
from r2lab_experiments.interference import csi_rb_power

band = PLAN.rbs(NOISE_FREQ, NOISE_RATE)
for component in CSI_COMPONENTS:
    fig, axes = plt.subplots(len(DIRECTIONS), 1, figsize=(10, 3.5 * len(DIRECTIONS)), squeeze=False)
    for ax, direction in zip(axes[:, 0], DIRECTIONS):
        for kind, color in colors.items():
            name = f"{direction}_{kind}"
            path = output_directory / "csi" / component / "by_window" / "section" / f"section__{name}" / "csi_per_rb.csv.gz"
            if not path.exists():
                print(f"{component}: no CSI for {name}")
                continue
            rbs, db, rows = csi_rb_power(path)
            ax.plot(rbs, db, color=color, linewidth=0.8, label=f"{kind} ({rows} rows)")
        if band:
            ax.axvspan(band[0], band[1], alpha=0.1, color="red")
        ax.set(title=f"{component}: {direction} transfer", xlabel="RB", ylabel="mean |H|² (dB)")
        ax.legend()
    plt.tight_layout()
    plt.show()

### Spectrum

With `SPECTRUM = "snapshots"`: the spectrum around the interference band in each section.

In [ ]:
for direction in DIRECTIONS:
    shown = {name.rsplit("_", 1)[-1]: s for name, s in spectra.items() if name.startswith(direction)}
    if shown:
        plot_spectra(shown, f"{direction} transfer: fit{SPECTRUM_FIT:02d} around {NOISE_FREQ / 1e6:.2f} MHz")

### Reading the results

- **Nothing changes**: the noise is too narrow, or does not reach the RU or the UE strongly enough. Widen it (an
  N320/N300 preset), raise `NOISE_GAIN` or `NOISE_AMPLITUDE`, or try another USRP (`NOISE_USRP`): the positions of
  the USRPs, the RU and the UE in the chamber set the levels.
- **The UE drops** (the transfer stops, the CU logs a release): the noise covers the SSB or the control channels
  (preset `ssb`), or it is too strong. The recovery section then shows the reattachment, if any.
- **Uplink only**: the noise reaches the RU more than the UE. **Downlink only**: the reverse.
- The **CSI** shows only the uplink, and only on the RBs covered: a tone (`sine`) hits one RB, the noise all of them.
- Run the scenario twice, with `INTERFERENCE = False` then `True`, to compare with a reference.

### End of the experiment

**Run All** stops at the next cell, so that the end-of-experiment cells below (stop of the live CSI views, UE
disconnection, release of the nodes) only run when you run them yourself, one at a time. The kernel keeps its state:
run the cells below with Shift+Enter when the experiment is over.

In [ ]:
class StopRunAll(Exception):
    """Ends "Run All" here; IPython shows only the message below (no traceback)."""

    def _render_traceback_(self):
        return [str(self)]


raise StopRunAll("Run All stops here. When the experiment is over, run the cells below one at a time.")

Stop the live CSI views before releasing the nodes: this closes the tunnels, stops the viewers and revokes their
SSH keys on the radio nodes.

In [ ]:
for view in globals().get("lives", {}).values():
    await view.stop()

Stop the interference tools: the emission (if any), the live spectrum, then power off the USRPs and the fit
nodes. The noise server is a POS node of the deployment model: the cleanup cell frees it with the RAN node.

In [ ]:
from r2lab_experiments.interference import power_n3xx, power_off_fit_node

if "noise" in globals() and noise.emitting:
    await noise.stop()
if globals().get("N3XX"):
    print(await power_n3xx(R2LAB_SLICE, NOISE_USRP, on=False))
if globals().get("live_spectrum") is not None:
    await live_spectrum.stop()
for fit in globals().get("FIT_NODES", []):
    print(await power_off_fit_node(R2LAB_SLICE, fit))

Let's disconnect the UEs for now:

In [ ]:
await ue.disconnect(force=True)

And finally save this experiment's meta model and deployment model:

In [ ]:
ue.output(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

> *Note*: All your Ansible run logs are stored by action, component and execution timestamp in `~/ifran/ansible-runner` by default, which you can of course change using the `runner_path` parameter in the `params.ansible` section of the deployment model.

## Cleanup — only after the experiment

Run the next cell **after** the experiment (and after stopping the live CSI views) to power off the RUs and the UEs (R2Lab radio API), free the POS node (allocation freed, calendar event kept) and destroy the VMs. The cell does nothing until you set `CONFIRM_CLEANUP = True` in it: the VMs hold the Kubernetes control plane, the core and its database, and the monitoring.

In [ ]:
# True: power off the RUs and the UEs, free the POS nodes, destroy the VMs (end of the experiment)
CONFIRM_CLEANUP = False
if not CONFIRM_CLEANUP:
    raise RuntimeError(
        "Cleanup not confirmed: set CONFIRM_CLEANUP = True in this cell to power off the RUs and UEs, free the POS"
        f" nodes and destroy {VM_NODES}"
    )
off = ProvisioningManagement(preceding=ue)
failed = {}
# RUs and UEs: powered off through the R2Lab radio API
for kind, power_off in (("RUs", off.unprovision_rus), ("UEs", off.unprovision_ues)):
    try:
        await power_off(quiet=True)
        print(f"{kind} powered off: {list(getattr(off.deployment_model, kind.lower()))}")
    except Exception as exc:  # noqa: BLE001 - go on with the rest, then fail
        failed[kind] = exc
# POS nodes, one at a time: allocation freed (slices pos allocations free <node> --keep-calendar-event)
for node, data in off.deployment_model.nodes.items():
    if data.provisioner != "pos" or not data.is_provisioned:
        continue
    try:
        await off.unprovision_nodes(nodes=[node], quiet=True)
        print(f"{node}: freed")
    except Exception as exc:  # noqa: BLE001 - report every node, then fail
        failed[node] = exc
# VMs, one at a time: destroyed in parallel, the "slices project use" calls of IFRAN race on the cache of the slices
# CLI (~/.slices/cache) and fail with FileNotFoundError. A destroyed VM is skipped: rerun the cell after a failure.
for node in VM_NODES:
    if not off.deployment_model.nodes[node].is_provisioned:
        print(f"{node}: already destroyed")
        continue
    try:
        await off.unprovision_nodes(nodes=[node], quiet=True)
        print(f"{node}: destroyed")
    except Exception as exc:  # noqa: BLE001 - report every VM, then fail
        failed[node] = exc
if failed:
    raise RuntimeError(f"Cleanup incomplete (run the cell again): {failed}")